# Bengali intent classification research notebook

See README.md and REPRODUCIBILITY.md before running. The supplied dataset/BnNERIntent25.json is loaded by default. Launch Jupyter from the repository root. Saved outputs were cleared for sharing; experimental training logic is preserved.

Google Drive mounting is optional in Colab. Install requirements.txt in your chosen environment before running the code cells.

In [ ]:
import random
import numpy as np
import torch
import json
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd
from transformers import AutoTokenizer
import evaluate
import numpy as np
from transformers import AutoModelForSequenceClassification,BertForSequenceClassification, TrainingArguments, Trainer


In [ ]:
# Set random values
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_val)

In [ ]:
# Load the exact original experiment data; do not substitute the 30-class collection.
import os
from pathlib import Path
DIRECTORY_PATH = Path(os.environ.get("BNNER_DATASET_PATH", "dataset/BnNERIntent25.json"))
if not DIRECTORY_PATH.is_file():
    raise FileNotFoundError(
        f"Missing experiment dataset: {DIRECTORY_PATH}. See dataset/README.md. "
        "Provide the original BnNERIntent25.json or set BNNER_DATASET_PATH."
    )
with DIRECTORY_PATH.open("r", encoding="utf-8") as file:
    data = json.load(file)

In [ ]:
# @title splitting it into train, validation, and test
splits = ["train", "val", "test"]
datasets = {}
labels = set()

for split in splits:
    datasets[split] = [{"text": item[0], "label": item[1]} for item in data[split]]
    labels.update([item[1] for item in data[split]])

# Convert to a Hugging Face Dataset
datasets = DatasetDict({
    split: Dataset.from_list(datasets[split]) for split in datasets.keys()
})

In [ ]:
datasets

In [ ]:
datasets.keys()

In [ ]:
#Preparing labels for classification and mapping them to unique IDs
label2id = {label: id for id, label in enumerate(labels)}
id2label = {id: label for label, id in label2id.items()}
len(labels)

In [ ]:
# @title Calculate statistics for the dataset
from collections import Counter
from datasets import DatasetDict
import re

# Function to calculate statistics for Bengali text
def dataset_statistics(dataset: DatasetDict):
    # Initialize variables
    total_words = 0
    word_counts = Counter()
    lengths = []

    # Iterate over the splits in the dataset
    for split in datasets:
        for text in datasets[split]['text']:
            # Tokenize the Bengali text using whitespace and punctuation as delimiters
            words = re.findall(r'\b\w+\b', text)
            # Update counters
            total_words += len(words)
            word_counts.update(words)
            lengths.append(len(words))

    # Calculate statistics
    total_unique_words = len(word_counts)
    max_length = max(lengths)
    min_length = min(lengths)
    average_length = total_words / len(lengths)

    # Print results
    print(f"Total Words: {total_words}")
    print(f"Total Unique Words: {total_unique_words}")
    print(f"Maximum Length: {max_length}")
    print(f"Minimum Length: {min_length}")
    print(f"Average Length: {average_length:.2f}")

# Calculate statistics for the dataset
dataset_statistics(datasets)


In [ ]:
# @title Model Initialization
from transformers import AutoTokenizer, pipeline
model_name = "sagorsarker/bangla-bert-base"
#model_name = "csebuetnlp/banglabert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(
    model_name, num_labels=len(labels), id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True)

intent_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer)


In [ ]:
import evaluate
import numpy as np

# Load metrics
accuracy_metric = evaluate.load('accuracy')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')
f1_metric = evaluate.load('f1')
mcc_metric = evaluate.load('matthews_correlation')  # Load the MCC metric

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    # Calculate the metrics
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    prec_macro = precision_metric.compute(predictions=predictions, references=labels, average='macro')
    rec_macro = recall_metric.compute(predictions=predictions, references=labels, average='macro')
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average='macro')
    mcc = mcc_metric.compute(predictions=predictions, references=labels)  # Compute MCC

    # Add computed metrics to the dictionary
    metrics = {
        'accuracy': acc['accuracy'],
        'precision_macro': prec_macro['precision'],
        'recall_macro': rec_macro['recall'],
        'f1_macro': f1_macro['f1'],
        'mcc': mcc['matthews_correlation'],
    }

    return metrics


In [ ]:
def preprocess_function(examples):
    text = examples["text"]
    labels = examples["label"]

    tokenized_text = tokenizer(text, truncation=False)
    labels = [label2id[label] for label in labels]

    return {"input_ids": tokenized_text["input_ids"],
            "attention_mask": tokenized_text["attention_mask"],
            "label": labels}

tokenized_datasets = datasets.map(preprocess_function, batched=True)

In [ ]:

from tabulate import tabulate

# Get a list of unique intents
unique_intents = set(tokenized_datasets['train']['label'])

# Prepare data for the table
table_data = []
for intent in unique_intents:
    train_count = sum(1 for example in tokenized_datasets['train'] if example['label'] == intent)
    test_count = sum(1 for example in tokenized_datasets['test'] if example['label'] == intent)
    val_count = sum(1 for example in tokenized_datasets['val'] if example['label'] == intent)
    table_data.append([id2label[intent], train_count, test_count, val_count])

# Print the table
headers = ["Intent", "Train Count", "Test Count", "Val Count"]
print(tabulate(table_data, headers=headers, tablefmt="grid"))

In [ ]:
#tokenized_datasets["val"][1]

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import EarlyStoppingCallback
task_name = "intent_classification"
output_dir = f"results/{model_name}_{task_name}/"

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,  # Lower learning rate
    per_device_train_batch_size=32,  # Smaller batch size
    per_device_eval_batch_size=32,
    num_train_epochs=10,  # Increase epochs
    weight_decay=0.01,  # Higher weight decay
    evaluation_strategy="epoch",  # Evaluate more frequently
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=500,
    load_best_model_at_end=True,
    save_total_limit=1,
    report_to="none",
    gradient_accumulation_steps=1,  # To maintain an effective batch size
    lr_scheduler_type="linear",
    warmup_steps=200,  # Increase warmup steps
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # Access EarlyStoppingCallback through trainer
)

trainer.train()


In [ ]:
# Evaluate on validation dataset
eval_metrics = trainer.evaluate(eval_dataset=tokenized_datasets["val"])
print("Validation Metrics:", eval_metrics)

# Predict on the test dataset
predictions = trainer.predict(tokenized_datasets["test"])
print("Test Metrics:", predictions.metrics)

In [ ]:
texts = [
    "ঢাকা থেকে দিনাজপুরের জন্য একটি ফ্লাইট বুক করতে চাই"
]

inputs = tokenizer(texts, return_tensors="pt", padding=True)
model = model.to('cpu') # put model to cpu
logits = model(**inputs).logits
print([model.config.id2label[item.item()] for item in logits.argmax(axis=-1)])

In [ ]:
# @title misclassified samples
# Evaluate the model
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model.to(device)
predictions = trainer.predict(tokenized_datasets["test"])
preds = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1)
pred_labels = torch.argmax(preds, axis=1)

# Identify misclassified samples
misclassified_indices = [i for i, (true, pred) in enumerate(zip(tokenized_datasets["test"]['label'], pred_labels)) if true != pred]

# Extract misclassified samples with probabilities
misclassified_samples = []
for idx in misclassified_indices:
    predicted_prob = preds[idx][pred_labels[idx]].item()
    sorted_probs, sorted_indices = torch.sort(preds[idx], descending=True)
    nearest_prob = sorted_probs[1].item() # This is the probability of the nearest probable class
    nearest_class = sorted_indices[1].item()

    sample = {
        'text': tokenized_datasets["test"]['text'][idx],
        'true_label': tokenized_datasets["test"]['label'][idx],
        'predicted_label': pred_labels[idx].item(),
        'predicted_prob': predicted_prob,
        'nearest_class': nearest_class,
        'nearest_prob': nearest_prob
    }
    misclassified_samples.append(sample)

# Print misclassified samples
for sample in misclassified_samples:
    print(f"Text: {sample['text']}")
    print(f"True Label: {sample['true_label']}")
    print(f"Predicted Label: {sample['predicted_label']}")
    print(f"Predicted Probability: {sample['predicted_prob']}")
    print(f"Nearest Probable Class: {sample['nearest_class']}")
    print(f"Probability of Nearest Probable Class: {sample['nearest_prob']}") # Print the probability of the nearest probable class
    print("\n")

In [ ]:
print(f'{id2label[18]=}')
print(f'{id2label[22]=}')
print(f'{id2label[2]=}')

In [ ]:
from transformers import TrainerCallback

class MetricsTrackerCallback(TrainerCallback):
    def __init__(self):
        self.metrics = {'train_loss': [], 'train_accuracy': [], 'eval_loss': [], 'eval_accuracy': [], 'epoch': []} # add 'epoch' key here

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if 'loss' in logs:
                self.metrics['train_loss'].append(logs['loss'])
            if 'eval_loss' in logs:
                self.metrics['eval_loss'].append(logs['eval_loss'])
            if 'eval_accuracy' in logs:
                self.metrics['eval_accuracy'].append(logs['eval_accuracy'])
            if 'eval_runtime' in logs:
                self.metrics['epoch'].append(state.epoch)

metrics_tracker = MetricsTrackerCallback()

In [ ]:
# @title Comparision for different BERT variants
model_names = [
    "csebuetnlp/banglabert",
    "sagorsarker/bangla-bert-base",
    "distilbert/distilbert-base-multilingual-cased",
    "csebuetnlp/banglat5"

]

results = []
from transformers import EarlyStoppingCallback
task_name = "intent_classification"
output_dir = f"results/{model_name}_{task_name}/"

for model_name in model_names:
    print(f"Training model {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    tokenized_datasets = datasets.map(preprocess_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = BertForSequenceClassification.from_pretrained(
        model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True)

    training_args = TrainingArguments(
        output_dir=f"results/{model_name}_{task_name}/",
        learning_rate=2e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=10,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_dir='./logs',
        logging_steps=500,
        load_best_model_at_end=True,
        save_total_limit=1,
        report_to="none",
        gradient_accumulation_steps=1,
        lr_scheduler_type="linear",
        warmup_steps=200,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["val"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3), metrics_tracker]
    )
    # Train the model and store the history
    train_result = trainer.train()
    history = train_result.metrics
    # Evaluate after training to get eval metrics
    eval_results = trainer.evaluate()
    # Add eval metrics to history
    history["eval_loss"] = eval_results.get("eval_loss", None)
    history["eval_accuracy"] = eval_results.get("eval_accuracy", None)
    # Predict on test set
    predictions = trainer.predict(tokenized_datasets["test"]).metrics

    results.append({
        "model_name": model_name,
        "history": history,
        "validation_metrics": trainer.evaluate(),
        "test_metrics": predictions,
        "metrics_tracker": metrics_tracker
    })

In [ ]:
# Print the results
for result in results:
    print(f"Model: {result['model_name']}, Validation Metrics (Trainer Metrics): {result['validation_metrics']}, Test Metrics: {result['test_metrics']}")  # Access with correct key

In [ ]:
# @title Calculate MAE, MSE, RMSE
# Evaluate after training to get eval metrics
eval_results = trainer.evaluate() # removed indentation
history["eval_loss"] = eval_results.get("eval_loss", None)
history["eval_accuracy"] = eval_results.get("eval_accuracy", None)

    # Predict on test set
predictions = trainer.predict(tokenized_datasets["test"]).predictions
predicted_labels = predictions.argmax(axis=1)
true_labels = tokenized_datasets["test"]["label"]

    # Calculate MAE, MSE, RMSE
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(true_labels, predicted_labels)
mse = mean_squared_error(true_labels, predicted_labels)
rmse = np.sqrt(mse)

print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")

results.append({
        "model_name": model_name,
        "history": history,
        "validation_metrics": trainer.evaluate(),
        "test_metrics": trainer.predict(tokenized_datasets["test"]).metrics,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "metrics_tracker": metrics_tracker
    })

In [ ]:
results

In [ ]:
import pandas as pd

# Extract model names
model_names = [model["model_name"] for model in results]

# Flatten validation and test metrics into a single dictionary for each model
flattened_data = []
for model in results:
    # Start with model name to ensure alignment
    model_data = {"model_name": model["model_name"]}

    # Add validation metrics
    for key, value in model["validation_metrics"].items():
        model_data[key] = value

    # Add test metrics
    for key, value in model["test_metrics"].items():
        model_data[key] = value

    flattened_data.append(model_data)

# Convert the list of dictionaries into a DataFrame
df = pd.DataFrame(flattened_data)

# Optionally, set the model name as the index of the DataFrame
df.set_index("model_name", inplace=True)

df.round(3).transpose()

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification,pipeline

In [ ]:
# Load a pre-trained NER model and tokenizer for Bengali (if available, or use multilingual)
bn_model_name = "Davlan/xlm-roberta-large-ner-hrl"  # example multilingual NER model
bn_tokenizer = AutoTokenizer.from_pretrained(bn_model_name)
bn_model = AutoModelForTokenClassification.from_pretrained(bn_model_name)
# Create pipelines for NER
bn_ner_pipeline = pipeline("ner", model=bn_model, tokenizer=bn_tokenizer)

In [ ]:
# Create the NER pipeline
bn_ner_pipeline = pipeline("ner", model=bn_model, tokenizer=bn_tokenizer)

# Select the first 50 sentences from the training set
bn_sentences = datasets['train']['text'][:10]

# Apply NER tagging to each sentence
ner_results = [bn_ner_pipeline(sentence) for sentence in bn_sentences]

# Display the NER results for each sentence
for i, (sentence, ner) in enumerate(zip(bn_sentences, ner_results)):
    print(f"Sentence {i+1}: {sentence}")
    print("NER Tags:", ner)
    print("\n")

In [ ]:
def process_text(text):
    # Predict the intent of the text
    intent_result = intent_pipeline(text)

    # Perform NER tagging
    ner_result = bn_ner_pipeline(text)

    return intent_result, ner_result


In [ ]:
# Example Bengali text input
bangla_text = " ঢাকা থেকে দিনাজপুরের জন্য একটি ফ্লাইট বুক করতে চাই।"

# Process the text
intent, ner_tags = process_text(bangla_text)

# Display the results
print("Intent of the text:", intent)
print("NER Tags:")
for entity in ner_tags:
    print(entity)


In [ ]:
def merge_tokens(ner_results):
    merged_tokens = []
    current_token = ""
    current_entity = None

    for result in ner_results:
        token = result['word']
        entity = result['entity']

        if token.startswith("▁"):
            if current_token:
                merged_tokens.append({'word': current_token, 'entity': current_entity})
            current_token = token[1:]
            current_entity = entity
        else:
            if entity == current_entity:
                current_token += token
            else:
                if current_token:
                    merged_tokens.append({'word': current_token, 'entity': current_entity})
                current_token = token
                current_entity = entity

    if current_token:
        merged_tokens.append({'word': current_token, 'entity': current_entity})

    return merged_tokens

# Example usage
bangla_text = "মাহিন দিনাজপুর এ অবস্থিত হাবিপ্রবি বিশ্ববিদ্যালয়ে পড়াশুনা করে।"
ner_tags = process_text(bangla_text)

# Display the results
print("NER Tags:")
for entity in ner_tags:
    print(entity)
